In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import time
from pathlib import Path
import sys

# ==========================================
# 0. PARAMÈTRES DE TON ÉQUIPE ET COLAB
# ==========================================
# --- DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT ---
if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy/"
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

# Imports du projet (Désormais possibles grâce à sys.path)
from mpp_project.core import apply_heteroscedastic_noise, apply_temporal_drift, calculate_true_outcome_probas_from_odds, estimate_crowd_3D
from mpp_project.v5_supervised.oracle_dp import compute_alphas_isolement, solve_dp_with_full_empirical_distribution, compute_full_Q_table, GAP_MIN, GAP_MAX, GAP_OFFSET

# Paramètres de jeu
MATCH_DU_JOUR_ID = 1    
MON_GAP_1 = 0
MON_GAP_2 = 0
HAS_BOOSTER = 1  # 1 = Booster disponible, 0 = Booster déjà utilisé
HORIZON_NUIT = 5
SEUIL_ISOLEMENT = 80.0  # Le réglage parfait de la variance (Calibration)

match_idx = MATCH_DU_JOUR_ID - 1

# ==========================================
# 1. PRÉPARATION DES DONNÉES DU JOUR
# ==========================================
print("Chargement des données du tournoi...")
df = pd.read_csv(f"{PROJECT_DIR}data/CDM_2026.csv")
match_phases = df['phase'].tolist()
n_matches = len(df)

# Probabilités réelles (purifiées de la marge bookmaker)
odds = df[['cote_1', 'cote_N', 'cote_2']].values.astype(np.float64)
base_true_probas = calculate_true_outcome_probas_from_odds(odds)

# Crowds (Répartitions) et Gains
base_crowds = df[['crowd_1', 'crowd_N', 'crowd_2']].values.astype(np.float64)
base_crowds = base_crowds / base_crowds.sum(axis=1, keepdims=True)

gains_1N2 = df[['gain_mpp_1', 'gain_mpp_N', 'gain_mpp_2']].values.astype(np.int32)

# Bruit bayésien
dists = np.arange(n_matches) - match_idx
dists_positive = np.maximum(0, dists)
alphas_bayes = 0.95 * (0.5 ** (dists_positive / 5.0))
alphas_2d = alphas_bayes.astype(np.float32)[:, np.newaxis]

c1, cN, c2 = estimate_crowd_3D(base_true_probas[:, 0], base_true_probas[:, 1], base_true_probas[:, 2], add_noise=False)
theo_crowds_pure = np.column_stack((c1, cN, c2)).astype(np.float32)
blended_mean_crowds = (alphas_2d * base_crowds) + ((1.0 - alphas_2d) * theo_crowds_pure)
dynamic_rmse = 0.083 * (1.0 - alphas_2d)


# ==========================================
# 2. CHARGEMENT DE LA MATRICE EMPIRIQUE
# ==========================================
print("Chargement de la matrice du peloton...")
try:
    p_empirique_1D = np.load(PROJECT_DIR + "data/p_empirique_1D.npy")
    print("✅ Matrice chargée avec succès !")
except FileNotFoundError:
    print("⚠️ Fichier introuvable. Assurez-vous d'avoir fait tourner le Notebook 15.")

max_gain_dynamique = p_empirique_1D.shape[2]


# ==========================================
# 3. GÉNÉRATION DU MULTIVERS DE DRIFT
# ==========================================
# Pour l'Oracle de nuit, on utilise directement les probabilités blendées :
base_true_probas = apply_temporal_drift(base_true_probas, match_phases, current_match_idx=match_idx)
base_crowds = apply_heteroscedastic_noise(blended_mean_crowds, rmse=dynamic_rmse)


# ==========================================
# 4. RÉSOLUTION DE LA PROGRAMMATION DYNAMIQUE
# ==========================================
data_path_npy = f"{PROJECT_DIR}data/expected_V_phases_finales.npy"
V_phases_finales_brut = np.load(data_path_npy)

nb_poules = len(df[df['phase'].str.contains('Poule', na=False)]) 
last_csv_match_id = df['match_id'].max()
npy_index = last_csv_match_id - nb_poules
V_next_base = V_phases_finales_brut[npy_index]

print("\n1. Calcul de la thermodynamique du peloton (Alphas)...")
alphas_isolement = compute_alphas_isolement(base_true_probas, base_crowds, gains_1N2, seuil_isolement=SEUIL_ISOLEMENT)

print("2. Lancement de l'Oracle DP (Rétro-propagation pure)...")
start_dp = time.time()

# 🚀 NOUVEAU : Résolution avec la matrice 1D et le Gating Temporel
V_all_matches = solve_dp_with_full_empirical_distribution(
    base_true_probas=base_true_probas, 
    base_crowds=base_crowds, 
    gains_1N2=gains_1N2, 
    p_empirique_1D=p_empirique_1D,
    alphas_isolement=alphas_isolement,
    V_horizon=V_next_base,
    stop_t=match_idx
)
print(f"-> Résolution DP terminée en {time.time() - start_dp:.2f} s.")


# ==========================================
# 5. EXTRACTION ET EXPORT DE L'ABAQUE DE DÉCISION (Q-TABLE)
# ==========================================
print("\n" + "="*50)
print(f"Génération de l'Abaque pour téléphone...")
print("="*50)

for k in range(HORIZON_NUIT):
    t = match_idx + k
    
    if t >= n_matches: break
        
    match_id_cible = t + 1
    
    if t + 1 < len(V_all_matches):
        V_next_t = V_all_matches[t + 1]
    else:
        V_next_t = V_next_base 

    t_prob = base_true_probas[t]
    c_rep = base_crowds[t]
    t_gains = gains_1N2[t]
    alpha_t = alphas_isolement[t]

    print(f"Calcul de l'Abaque pour le Match {match_id_cible}...")
    Q_table_t = compute_full_Q_table(
        t=t,
        t_prob=t_prob, 
        c_rep=c_rep, 
        t_gains=t_gains, 
        p_empirique_1D=p_empirique_1D,
        alpha=alpha_t,
        V_next=V_next_t, 
        max_gain=max_gain_dynamique
    )

    export_path = f"{PROJECT_DIR}data/Abaque_Match_{match_id_cible}.npz"
    np.savez_compressed(export_path, q_table=Q_table_t, ev_actions=t_prob * t_gains)
    
print(f"\n✅ Terminé ! {HORIZON_NUIT} fichiers synchronisés sur le Drive.")
print("Le notebook '17_night_mode.ipynb' est désormais utilisable depuis le téléphone.")


# ==========================================
# 6. ÉVALUATION STRATÉGIQUE DU MATCH ACTUEL
# ==========================================
if match_idx + 1 < len(V_all_matches):
    V_next = V_all_matches[match_idx + 1]
else:
    V_next = V_next_base

t_prob = base_true_probas[match_idx]
c_rep = base_crowds[match_idx]
t_gains = gains_1N2[match_idx]
alpha = alphas_isolement[match_idx]  # 🚀 LECTURE DU ALPHA POUR LE MATCH DU JOUR

wr_keep = np.zeros(3)
wr_use = np.zeros(3)
ev_actions = np.zeros(3)

for a in range(3):
    ev_actions[a] = t_prob[a] * t_gains[a]
    expected_v_b0 = 0.0
    expected_v_keep = 0.0
    expected_v_use = 0.0
    
    for out in range(3):
        p_out = t_prob[out]
        if p_out == 0.0: continue
            
        a_g = t_gains[out] if a == out else 0
        a_g_boost = (t_gains[out] * 2) if a == out else 0
        true_gain = t_gains[out]
        out_p_empirique = p_empirique_1D[match_idx, out]
            
        for bob_a in range(3):
            p_bob = c_rep[bob_a]
            if p_bob == 0.0: continue
                
            bob_g = true_gain if bob_a == out else 0
            jp_base = p_out * p_bob
            
            new_gap_bob_norm = MON_GAP_1 + a_g - bob_g
            new_gap_bob_boost = MON_GAP_1 + a_g_boost - bob_g
            
            esp_peloton_b0 = 0.0
            esp_peloton_keep = 0.0
            esp_peloton_use = 0.0
            
            for delta_gain in range(max_gain_dynamique):
                p_peloton = out_p_empirique[delta_gain]
                if p_peloton == 0.0: continue
                    
                new_gap_peloton_norm = MON_GAP_2 + a_g - delta_gain
                new_gap_peloton_boost = MON_GAP_2 + a_g_boost - delta_gain
                
                # 🚀 LA MAGIE DU GATING TEMPOREL 🚀
                val_g1_norm = min(new_gap_bob_norm, new_gap_peloton_norm)
                val_g2_norm = alpha * max(new_gap_bob_norm, new_gap_peloton_norm) + (1.0 - alpha) * new_gap_peloton_norm
                
                val_g1_boost = min(new_gap_bob_boost, new_gap_peloton_boost)
                val_g2_boost = alpha * max(new_gap_bob_boost, new_gap_peloton_boost) + (1.0 - alpha) * new_gap_peloton_boost
                
                g1_norm = max(GAP_MIN, min(GAP_MAX, int(round(val_g1_norm)))) + GAP_OFFSET
                g2_norm = max(GAP_MIN, min(GAP_MAX, int(round(val_g2_norm)))) + GAP_OFFSET
                g1_boost = max(GAP_MIN, min(GAP_MAX, int(round(val_g1_boost)))) + GAP_OFFSET
                g2_boost = max(GAP_MIN, min(GAP_MAX, int(round(val_g2_boost)))) + GAP_OFFSET
                
                esp_peloton_b0 += p_peloton * V_next[g1_norm, g2_norm, 0]
                esp_peloton_keep += p_peloton * V_next[g1_norm, g2_norm, 1]
                esp_peloton_use += p_peloton * V_next[g1_boost, g2_boost, 0]
                
            expected_v_b0 += jp_base * esp_peloton_b0
            expected_v_keep += jp_base * esp_peloton_keep
            expected_v_use += jp_base * esp_peloton_use

    if HAS_BOOSTER:
        wr_keep[a] = expected_v_keep
        wr_use[a] = expected_v_use
    else:
        wr_keep[a] = expected_v_b0
        wr_use[a] = 0.0

# --- AFFICHAGE ---
print(f"\n--- MATCH {MATCH_DU_JOUR_ID} : ANALYSE DES CHOIX ---")
noms_choix = ["1 (Domicile)", "N (Nul)", "2 (Extérieur)"]

if HAS_BOOSTER:
    for i in range(3):
        print(f"Choix {noms_choix[i]:<14} : WR (Garder x2) = {wr_keep[i]*100:05.2f}% | WR (Utiliser x2) = {wr_use[i]*100:05.2f}% | EV = {ev_actions[i]*2:.2f}")
    
    best_keep_idx = np.argmax(wr_keep)
    best_use_idx = np.argmax(wr_use)
    
    if wr_use[best_use_idx] > wr_keep[best_keep_idx]:
        gain_wr = (wr_use[best_use_idx] - wr_keep[best_keep_idx]) * 100
        print(f"\n🚀 RECOMMANDATION ROBUSTE : Jouer le {noms_choix[best_use_idx]} ET UTILISER LE BOOSTER x2 (Gain WR: +{gain_wr:.2f}%)")
    else:
        print(f"\n✅ RECOMMANDATION ROBUSTE : Jouer le {noms_choix[best_keep_idx]} (Garder le Booster pour plus tard)")
else:
    for i in range(3):
        print(f"Choix {noms_choix[i]:<14} : Win Rate = {wr_keep[i]*100:05.2f}% | EV = {ev_actions[i]:.2f} pts")
    best_action = np.argmax(wr_keep)
    print(f"\n✅ RECOMMANDATION ROBUSTE : Jouer le {noms_choix[best_action]}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
Chargement des données du tournoi...
Chargement de la matrice du peloton...
✅ Matrice chargée avec succès !

1. Calcul de la thermodynamique du peloton (Alphas)...
2. Lancement de l'Oracle DP (Rétro-propagation pure)...
-> Résolution DP terminée en 479.73 s.

Génération de l'Abaque pour téléphone...
Calcul de l'Abaque pour le Match 1...
Calcul de l'Abaque pour le Match 2...
Calcul de l'Abaque pour le Match 3...
Calcul de l'Abaque pour le Match 4...
Calcul de l'Abaque pour le Match 5...

✅ Terminé ! 5 fichiers synchronisés sur le Drive.
Le notebook '17_night_mode.ipynb' est désormais utilisable depuis le téléphone.

--- MATCH 1 : ANALYSE DES CHOIX ---
Choix 1 (Domicile)   : WR (Garder x2) = 33.45% | WR (Utiliser x2) = 29.55% | EV = 61.71
Choix N (Nul)        : WR (Garder x2) = 33.34% | WR (Utiliser x2) = 29.65% | EV = 57.59
Choix 2 (Extérieur)  : WR (Garder x2

In [5]:
# ==========================================
# 7. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

# On regarde sur 4 matchs maximum, ou jusqu'à la fin de l'horizon disponible
horizon_projection = min(4, HORIZON_NUIT, n_matches - match_idx)

for k in range(horizon_projection):
    t = match_idx + k
    match_id_cible = t + 1
    
    # Chargement de l'abaque précalculée à l'étape 5
    try:
        abaque_path = f"{PROJECT_DIR}data/Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        # Calcul des gaps projetés
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        # Normalisation pour l'index de la matrice
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        # Récupération de la meilleure action
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            
            best_keep_idx = np.argmax(wr_keep)
            best_use_idx = np.argmax(wr_use)
            
            val_keep = wr_keep[best_keep_idx]
            val_use = wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
                
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        # Affichage formaté
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 1 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.45% | WR(x2): 29.65%
  ⚪ Maintien (Inchangé)       | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.45% | WR(x2): 29.65%
  🟢 Avance (+60 pts/match)    | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.45% | WR(x2): 29.65%
------------------------------------------------------------------------------------------
▶️ MATCH 2 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :  -60,  -60 | Reco : 1 (Dom) (Safe) | WR(Safe): 31.20% | WR(x2): 27.76%
  ⚪ Maintien (Inchangé)       | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 35.18% | WR(x2): 31.89%
  🟢 Avance (+60 pts/match)    | Gaps proj. :   60,   60 | Reco : 1 (Dom) (Safe) | WR(Safe): 39.74% | WR(x2): 36.57%
-------------------------------